In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import zscore

In [11]:
df = pd.read_csv('../data/sudan.csv', encoding='latin1', skiprows=[1])
df['Country'] = 'Sudan'
print(df.shape)
df.head()

(4107, 13)


,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country
0,2015,2,23.92,34.14,15.81,18.33,0.0,23.83,4.24,5.10,96.67,4.31,Sudan
1,2015,3,22.73,31.64,15.09,16.55,0.0,38.21,5.01,6.24,96.77,6.42,Sudan
2,2015,4,19.15,27.35,12.88,14.47,0.0,21.07,5.72,6.96,96.93,3.06,Sudan
3,2015,5,17.54,27.22,9.49,17.73,0.0,21.58,4.28,5.82,96.85,2.86,Sudan
4,2015,6,17.18,25.67,10.15,15.52,0.0,25.86,4.46,6.07,96.96,3.11,Sudan


In [12]:
df["Date"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")
df['MONTH'] = df['Date'].dt.month
df.tail()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Country,Date,MONTH
4102,2026,86,27.89,34.69,21.67,13.02,0.0,14.47,3.66,5.11,96.47,3.52,Sudan,2026-03-27,3
4103,2026,87,28.73,36.61,21.34,15.27,0.0,9.75,3.21,4.40,96.35,2.44,Sudan,2026-03-28,3
4104,2026,88,30.06,38.89,21.15,17.74,0.0,11.63,2.41,3.87,96.10,3.12,Sudan,2026-03-29,3
4105,2026,89,32.50,41.53,23.61,17.92,0.0,14.06,2.81,4.87,95.93,4.18,Sudan,2026-03-30,3
4106,2026,90,33.79,42.64,25.73,16.91,0.0,13.89,3.52,5.00,95.89,4.47,Sudan,2026-03-31,3


In [13]:
# Check for missing values and duplicates(-999 is NASA's sentinel value for missing or out-of-range data.)
df = df.replace(-999, np.nan)
print("Number of duplicate rows found:", df.duplicated().sum())
df = df.drop_duplicates()

Number of duplicate rows found: 0


In [14]:
df.describe()

,YEAR,DOY,T2M,T2M_MAX,T2M_MIN,T2M_RANGE,PRECTOTCORR,RH2M,WS2M,WS2M_MAX,PS,QV2M,Date,MONTH
count,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107.000000,4107,4107.000000
mean,2020.132700,180.164841,28.759878,36.774212,21.509817,15.264395,0.644032,31.359067,3.484259,5.157387,96.346226,7.864991,2020-08-16 00:00:00,6.424884
min,2015.000000,1.000000,13.180000,21.040000,5.930000,3.420000,0.000000,4.690000,0.610000,1.030000,95.660000,1.160000,2015-01-02 00:00:00,1.000000
25%,2017.000000,86.000000,25.560000,33.730000,17.500000,13.280000,0.000000,17.405000,2.730000,4.265000,96.160000,3.730000,2017-10-24 12:00:00,3.000000
50%,2020.000000,179.000000,29.160000,37.020000,22.890000,15.810000,0.000000,26.630000,3.490000,5.120000,96.310000,5.890000,2020-08-16 00:00:00,6.000000
75%,2023.000000,272.000000,32.510000,40.330000,25.430000,17.680000,0.010000,40.560000,4.220000,6.020000,96.510000,12.500000,2023-06-08 12:00:00,9.000000
max,2026.000000,366.000000,37.990000,45.960000,32.170000,22.480000,66.490000,87.160000,7.150000,9.050000,97.310000,19.440000,2026-03-31 00:00:00,12.000000
std,3.248315,106.270943,4.681542,4.400559,5.091072,3.298687,3.058028,17.854021,1.040793,1.280393,0.266942,4.881449,NaN,3.476439


In [15]:
df.isna().sum() 
missing_percent = (df.isna().sum() / len(df)) * 100
print(missing_percent)

YEAR           0.0
DOY            0.0
T2M            0.0
T2M_MAX        0.0
T2M_MIN        0.0
T2M_RANGE      0.0
PRECTOTCORR    0.0
RH2M           0.0
WS2M           0.0
WS2M_MAX       0.0
PS             0.0
QV2M           0.0
Country        0.0
Date           0.0
MONTH          0.0
dtype: float64


In [16]:
cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]

z_scores = df[cols].apply(zscore)
outliers = (np.abs(z_scores) > 3)
outlier_rows = outliers.any(axis=1).sum()
print("Number of rows with extreme values (|Z| > 3):", outlier_rows)

Number of rows with extreme values (|Z| > 3): 84


In [17]:
# Forward-fill weather-related columns
weather_cols = ["T2M", "T2M_MAX", "T2M_MIN", "PRECTOTCORR", "RH2M", "WS2M", "WS2M_MAX"]
df[weather_cols] = df[weather_cols].ffill()
threshold = int(0.7 * df.shape[1])  # keep rows with at least 70% non-null values
df = df.dropna(thresh=threshold)


In [18]:
df.to_csv("../data/sudan_clean.csv", index=False)